# **Importing Libraries**

In [1]:
# Standard Library Imports
import os            # OS-level operations (paths, directory handling)
import math          # Mathematical functions
import shutil        # High-level file operations (copying, moving, deleting)

# Data Handling & Processing
import numpy as np               # Numerical computations and array operations
import pandas as pd              # Data manipulation and analysis
import h5py                      # Handling HDF5 file formats

# Progress Visualization
from tqdm import tqdm            # Progress bars for loops

# Plotting & Visualization
import matplotlib.pyplot as plt  # Plotting and visualization tools


# PyTorch Machine Learning Stack
import torch                     # Core PyTorch library
import torch.nn as nn            # Neural network layers and utilities
import torch.optim as optim      # Optimization algorithms (SGD, Adam, etc.)
from torch.utils.data import (   # Dataset and DataLoader utilities
    DataLoader,
    Dataset
)

# **Defining the architecture of sparse autoencoder**

In [2]:
"""
Sparse Autoencoder model.

Anthropic-style ReLU SAE with L1 sparsity penalty and unit-norm decoder
columns, following Bricken et al. (2023).
"""

class SparseAutoencoder(nn.Module):
    """ReLU sparse autoencoder for decomposing neural network activations.

    Args:
        d_input: Dimensionality of the input activations.
        d_hidden: Dictionary size (number of latent features).
    """

    def __init__(self, d_input: int, d_hidden: int):
        super().__init__()
        self.d_input = d_input
        self.d_hidden = d_hidden
        self.W_enc = nn.Linear(d_input, d_hidden, bias=True)
        self.W_dec = nn.Linear(d_hidden, d_input, bias=True)
        with torch.no_grad():
            self.W_dec.weight.data = nn.functional.normalize(
                self.W_dec.weight.data, dim=0
            )

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Encode input activations into sparse feature activations."""
        return torch.relu(self.W_enc(x - self.W_dec.bias))

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        """Decode sparse features back to activation space."""
        return self.W_dec(z)

    def forward(self, x: torch.Tensor):
        """Full forward pass: encode then decode.

        Returns:
            x_hat: Reconstructed activations.
            z: Sparse feature activations.
        """
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z

    def loss(self, x: torch.Tensor, lam: float):
        """Compute SAE loss = MSE + lambda * L1.

        Returns:
            total_loss, mse, l1, z
        """
        x_hat, z = self.forward(x)
        mse = (x - x_hat).pow(2).mean()
        l1 = z.abs().mean()
        return mse + lam * l1, mse, l1, z

    def normalize_decoder(self):
        """Constrain decoder column norms to unity (call after each step)."""
        with torch.no_grad():
            self.W_dec.weight.data = nn.functional.normalize(
                self.W_dec.weight.data, dim=0
            )

# **Defining custom dataset class for training on activations**

In [3]:
# Dataset class
class ActivationDataset(Dataset):
    def __init__(self, activation_path):

        self.x = torch.load(activation_path).float()

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx]

# **Define the training function**

In [4]:
# Training
def train_sae(
    activation_path,
    save_dir,
    d_hidden,
    lam=1,
    epochs=50,
    batch_size=512,
    lr=1e-3,
    device="cuda"
):

    # Make the directory to save the trained models
    os.makedirs(save_dir, exist_ok=True)

    # get the activation dataset
    dataset = ActivationDataset(activation_path)

    # Define the dataloader
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    # Finding the embedding dimension in input
    d_input = dataset[0].shape[0]

    # Displaying the input dimension and dictionary size
    print(f"Input dimension : {d_input}")
    print(f"Dictionary size : {d_hidden}")

    # Define the sparse autoencoder model
    sae = SparseAutoencoder(
        d_input=d_input,
        d_hidden=d_hidden
    ).to(device)

    # define the optimizer
    optimizer = torch.optim.Adam(
        sae.parameters(),
        lr=lr
    )

    # Initialize the best loss
    best_loss = 1e9

    # For each epoch
    for epoch in range(epochs):
        # Set the sae to train mode
        sae.train()
        # Initialize the running loss, mse and l1 sparsity penalty loss to zero
        running_loss = 0
        running_mse = 0
        running_l1 = 0
        # tqdm taskbar
        pbar = tqdm(loader)
        # for each sample
        for x in pbar:
            # move the sample to device
            x = x.to(device)
            # Forward pass through sae and get loss, mse, l1 and activations
            loss, mse, l1, z = sae.loss(
                x,
                lam=lam
            )
            # reset the optimizer
            optimizer.zero_grad()
            # backward pass
            loss.backward()
            # make the update
            optimizer.step()
            # normalize the decoder
            sae.normalize_decoder()
            # accumulate the losses
            running_loss += loss.item()
            running_mse += mse.item()
            running_l1 += l1.item()
            # display the progress
            pbar.set_description(
                f"Epoch {epoch+1}"
            )
            pbar.set_postfix(
                loss=loss.item(),
                mse=mse.item(),
                l1=l1.item()
            )

        # calculate the epoch loss
        epoch_loss = running_loss / len(loader)
        # Print the losses
        print(
            f"Epoch {epoch+1:03d} | "
            f"Loss={epoch_loss:.6f} | "
            f"MSE={running_mse/len(loader):.6f} | "
            f"L1={running_l1/len(loader):.6f}"
        )

        # Save the best model
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save(
                {
                    "model": sae.state_dict(),
                    "input_dim": d_input,
                    "hidden_dim": d_hidden,
                    "lambda": lam
                },

                os.path.join(
                    save_dir,
                    "best_sae.pt"
                )
            )
    print("Training Finished.")

# **Train separate SAE for each layer's activations (seed 0)**

In [5]:
# Train the sae on temporal layer activations
train_sae(
    "/kaggle/input/notebooks/sumanpunshi123/extract-activations/activations/seed_0/train/temporal.pt",
    "sae_temporal_seed0",
    d_hidden=512
)
# Train the sae on spatial layer activations
train_sae(
    "/kaggle/input/notebooks/sumanpunshi123/extract-activations/activations/seed_0/train/spatial.pt",
    "sae_spatial_seed0",
    d_hidden=512

)
# Train the sae on lstm layer activations
train_sae(
    "/kaggle/input/notebooks/sumanpunshi123/extract-activations/activations/seed_0/train/lstm.pt",
    "sae_lstm_seed0",
    d_hidden=1024

)
# Train the sae on pooled layer activations
train_sae(
    "/kaggle/input/notebooks/sumanpunshi123/extract-activations/activations/seed_0/train/pooled.pt",
    "sae_pooled_seed0",
    d_hidden=1024

)

Input dimension : 32
Dictionary size : 512


Epoch 1: 100%|██████████| 334/334 [00:02<00:00, 152.09it/s, l1=0.0132, loss=0.014, mse=0.000826]


Epoch 001 | Loss=0.054196 | MSE=0.015490 | L1=0.038707


Epoch 2: 100%|██████████| 334/334 [00:01<00:00, 192.49it/s, l1=0.00599, loss=0.00643, mse=0.00044]


Epoch 002 | Loss=0.009129 | MSE=0.000589 | L1=0.008539


...

Epoch 048 | Loss=0.002146 | MSE=0.000268 | L1=0.001877


Epoch 49: 100%|██████████| 334/334 [00:01<00:00, 193.51it/s, l1=0.00193, loss=0.00218, mse=0.000256]


Epoch 049 | Loss=0.002144 | MSE=0.000268 | L1=0.001876


Epoch 50: 100%|██████████| 334/334 [00:01<00:00, 190.52it/s, l1=0.00181, loss=0.00211, mse=0.000296]


Epoch 050 | Loss=0.002142 | MSE=0.000267 | L1=0.001876
Training Finished.


# **Train separate SAE for each layer's activations (seed 1)**

In [6]:
# Train the sae on temporal layer activations
train_sae(
    "/kaggle/input/notebooks/sumanpunshi123/extract-activations/activations/seed_1/train/temporal.pt",
    "sae_temporal_seed1",
    d_hidden=512
)
# Train the sae on spatial layer activations
train_sae(
    "/kaggle/input/notebooks/sumanpunshi123/extract-activations/activations/seed_1/train/spatial.pt",
    "sae_spatial_seed1",
    d_hidden=512

)
# Train the sae on lstm layer activations
train_sae(
    "/kaggle/input/notebooks/sumanpunshi123/extract-activations/activations/seed_1/train/lstm.pt",
    "sae_lstm_seed1",
    d_hidden=1024

)
# Train the sae on pooled layer activations
train_sae(
    "/kaggle/input/notebooks/sumanpunshi123/extract-activations/activations/seed_1/train/pooled.pt",
    "sae_pooled_seed1",
    d_hidden=1024

)

Input dimension : 32
Dictionary size : 512


Epoch 1: 100%|██████████| 334/334 [00:01<00:00, 199.65it/s, l1=0.00867, loss=0.00944, mse=0.000762]


Epoch 001 | Loss=0.042800 | MSE=0.011567 | L1=0.031233


Epoch 2: 100%|██████████| 334/334 [00:01<00:00, 195.98it/s, l1=0.00526, loss=0.00559, mse=0.00033]


Epoch 002 | Loss=0.006995 | MSE=0.000487 | L1=0.006508


...

Epoch 048 | Loss=0.004720 | MSE=0.000515 | L1=0.004205


Epoch 49: 100%|██████████| 334/334 [00:01<00:00, 193.55it/s, l1=0.00417, loss=0.00472, mse=0.000542]


Epoch 049 | Loss=0.004717 | MSE=0.000514 | L1=0.004203


Epoch 50: 100%|██████████| 334/334 [00:01<00:00, 195.28it/s, l1=0.00425, loss=0.0047, mse=0.000454]


Epoch 050 | Loss=0.004713 | MSE=0.000509 | L1=0.004203
Training Finished.


# **Train separate SAE for each layer's activations (seed 42)**

In [7]:
# Train the sae on temporal layer activations
train_sae(
    "/kaggle/input/notebooks/sumanpunshi123/extract-activations/activations/seed_42/train/temporal.pt",
    "sae_temporal_seed42",
    d_hidden=512
)
# Train the sae on spatial layer activations
train_sae(
    "/kaggle/input/notebooks/sumanpunshi123/extract-activations/activations/seed_42/train/spatial.pt",
    "sae_spatial_seed42",
    d_hidden=512

)
# Train the sae on lstm layer activations
train_sae(
    "/kaggle/input/notebooks/sumanpunshi123/extract-activations/activations/seed_42/train/lstm.pt",
    "sae_lstm_seed42",
    d_hidden=1024

)
# Train the sae on pooled layer activations
train_sae(
    "/kaggle/input/notebooks/sumanpunshi123/extract-activations/activations/seed_42/train/pooled.pt",
    "sae_pooled_seed42",
    d_hidden=1024

)

Input dimension : 32
Dictionary size : 512


Epoch 1: 100%|██████████| 334/334 [00:01<00:00, 193.39it/s, l1=0.00614, loss=0.00672, mse=0.00058]


Epoch 001 | Loss=0.030712 | MSE=0.007127 | L1=0.023585


Epoch 2: 100%|██████████| 334/334 [00:01<00:00, 194.78it/s, l1=0.00428, loss=0.00454, mse=0.00026]


Epoch 002 | Loss=0.004965 | MSE=0.000375 | L1=0.004590


...

Epoch 048 | Loss=0.006489 | MSE=0.000765 | L1=0.005724


Epoch 49: 100%|██████████| 334/334 [00:01<00:00, 184.10it/s, l1=0.00573, loss=0.00654, mse=0.000813]


Epoch 049 | Loss=0.006488 | MSE=0.000768 | L1=0.005720


Epoch 50: 100%|██████████| 334/334 [00:01<00:00, 186.33it/s, l1=0.00581, loss=0.00658, mse=0.000765]


Epoch 050 | Loss=0.006481 | MSE=0.000766 | L1=0.005716
Training Finished.


# **Train separate SAE for each layer's activations (seed 100)**

In [8]:
# Train the sae on temporal layer activations
train_sae(
    "/kaggle/input/notebooks/sumanpunshi123/extract-activations/activations/seed_100/train/temporal.pt",
    "sae_temporal_seed100",
    d_hidden=512
)
# Train the sae on spatial layer activations
train_sae(
    "/kaggle/input/notebooks/sumanpunshi123/extract-activations/activations/seed_100/train/spatial.pt",
    "sae_spatial_seed100",
    d_hidden=512

)
# Train the sae on lstm layer activations
train_sae(
    "/kaggle/input/notebooks/sumanpunshi123/extract-activations/activations/seed_100/train/lstm.pt",
    "sae_lstm_seed100",
    d_hidden=1024

)
# Train the sae on pooled layer activations
train_sae(
    "/kaggle/input/notebooks/sumanpunshi123/extract-activations/activations/seed_100/train/pooled.pt",
    "sae_pooled_seed100",
    d_hidden=1024

)

Input dimension : 32
Dictionary size : 512


Epoch 1: 100%|██████████| 334/334 [00:01<00:00, 197.36it/s, l1=0.00535, loss=0.00597, mse=0.000626]


Epoch 001 | Loss=0.029065 | MSE=0.005859 | L1=0.023205


Epoch 2: 100%|██████████| 334/334 [00:01<00:00, 195.84it/s, l1=0.00368, loss=0.00397, mse=0.000293]


Epoch 002 | Loss=0.004617 | MSE=0.000416 | L1=0.004201


...

Epoch 048 | Loss=0.007446 | MSE=0.001143 | L1=0.006303


Epoch 49: 100%|██████████| 334/334 [00:01<00:00, 191.37it/s, l1=0.00619, loss=0.00746, mse=0.00127]


Epoch 049 | Loss=0.007438 | MSE=0.001144 | L1=0.006293


Epoch 50: 100%|██████████| 334/334 [00:01<00:00, 193.73it/s, l1=0.0063, loss=0.00739, mse=0.00109]


Epoch 050 | Loss=0.007426 | MSE=0.001140 | L1=0.006285
Training Finished.
